# Tome C2, Chapter 5: Dynamic Programming (Python companion)

Exercises first, solutions at the end. Every dynamic programme of the chapter, implemented and checked against brute force: edit distance with Hirschberg's linear-space alignment, longest common and increasing subsequences (with a glimpse of a deep theorem about random permutations), knapsacks, matrix chains, optimal search trees with and without the Knuth–Yao speed-up, Viterbi decoding of a dishonest casino, the Held–Karp travelling salesman, independent sets in trees, and the line breaker of Exercise 5.6.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, math, time, itertools, bisect
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt
random.seed(5)

## Exercises

**P5.1 (★★) Edit distance and Hirschberg.** Implement Theorem 5.1 with traceback, and Hirschberg's linear-space alignment (Theorem 5.2). Check on random words that both give alignments of cost $\mathrm{ed}(a,b)$, and compare the memory used.

**P5.2 (★★) Subsequences.** Implement the LCS recurrence (Exercise 5.1) and the $O(n\log n)$ longest increasing subsequence (Exercise 5.3), checked against the quadratic DP. Then measure the longest increasing subsequence of random permutations of size $n$ and compare with $2\sqrt n$.

**P5.3 (★) Knapsack.** Implement Theorem 5.3 and Exercise 5.5 and compare with brute force.

**P5.4 (★) Matrix chains.** Implement Proposition 5.4, check it against all parenthesisations for small $n$, and look at the ratio between the worst and the best order on random dimensions.

**P5.5 (★★) Optimal search trees.** Implement the cubic recurrence (5.1) and the Knuth–Yao version of Theorem 5.6; check that they agree, that the roots $K(i,j)$ are monotone, and count the work.

**P5.6 (★★) The dishonest casino.** A casino uses a fair die and, now and then, a loaded one. Simulate it as a hidden Markov model, decode the hidden die with Viterbi (Theorem 5.7), and compute posterior probabilities with the forward algorithm (Exercise 5.9) and its backward twin.

**P5.7 (★) Held–Karp.** Implement Theorem 5.8, check it against brute force over all tours, and time it.

**P5.8 (★) Trees.** Implement Proposition 5.9 and Exercise 5.8 on random trees and check them against brute force.

**P5.9 (★) Breaking paragraphs.** Set a paragraph with the greedy method and with the dynamic programme of Exercise 5.6, and compare.

## Solutions

### P5.1 Edit distance and Hirschberg
An alignment is returned as a list of column pairs, with `'-'` for a gap.

In [ ]:
def edit_table(a, b):
    m, n = len(a), len(b)
    E = [[i + j if i == 0 or j == 0 else 0 for j in range(n + 1)] for i in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            E[i][j] = min(E[i-1][j] + 1, E[i][j-1] + 1, E[i-1][j-1] + (a[i-1] != b[j-1]))
    return E

def align_full(a, b):
    E = edit_table(a, b); i, j = len(a), len(b); cols = []
    while i or j:
        if i and j and E[i][j] == E[i-1][j-1] + (a[i-1] != b[j-1]): cols.append((a[i-1], b[j-1])); i -= 1; j -= 1
        elif i and E[i][j] == E[i-1][j] + 1: cols.append((a[i-1], '-')); i -= 1
        else: cols.append(('-', b[j-1])); j -= 1
    return cols[::-1]

def last_row(a, b):
    # E(len(a), j) for all j, in O(len(b)) space
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (a[i-1] != b[j-1]))
        prev = cur
    return prev

def align_hirschberg(a, b):
    if len(a) == 0: return [('-', y) for y in b]
    if len(a) == 1 or len(b) == 0: return align_full(a, b)    # O(len(b)) here
    h = len(a) // 2
    F = last_row(a[:h], b)
    B = last_row(a[h:][::-1], b[::-1])[::-1]
    j = min(range(len(b) + 1), key=lambda j: F[j] + B[j])
    return align_hirschberg(a[:h], b[:j]) + align_hirschberg(a[h:], b[j:])

cost = lambda cols: sum(x != y for x, y in cols)
def show(cols): print(''.join(x for x, _ in cols)); print(''.join(y for _, y in cols))
show(align_full("kitten", "sitting")); print("cost", cost(align_full("kitten", "sitting")))
show(align_hirschberg("dynamic", "programming"))

for trial in range(300):
    a = ''.join(random.choice('ACGT') for _ in range(random.randrange(0, 40)))
    b = ''.join(random.choice('ACGT') for _ in range(random.randrange(0, 40)))
    d = edit_table(a, b)[-1][-1]
    for cols in (align_full(a, b), align_hirschberg(a, b)):
        assert ''.join(x for x, _ in cols if x != '-') == a and ''.join(y for _, y in cols if y != '-') == b
        assert cost(cols) == d
print("full and linear-space alignments are optimal on 300 random DNA pairs")

In [ ]:
import tracemalloc
n = 1500
a = ''.join(random.choice('ACGT') for _ in range(n)); b = ''.join(random.choice('ACGT') for _ in range(n))
for name, fn in (("full table", align_full), ("Hirschberg", align_hirschberg)):
    tracemalloc.start(); t0 = time.perf_counter(); cols = fn(a, b); t = time.perf_counter() - t0
    peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
    print(f"{name:11s}: cost {cost(cols)}, {t:5.1f} s, peak memory {peak/1e6:6.1f} MB")
print(f"ed/n = {cost(cols)/n:.3f} for random DNA words")

Hirschberg's method does about twice the arithmetic in theory, yet here it is no slower (smaller rows are kinder to the memory system), and it saves almost all the memory. (Python lists of integers are wasteful; in C the full table for two words of a million letters would still be a terabyte.)

### P5.2 Subsequences

In [ ]:
def lcs(a, b):
    m, n = len(a), len(b); L = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            L[i][j] = L[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(L[i-1][j], L[i][j-1])
    out, i, j = [], m, n
    while i and j:
        if a[i-1] == b[j-1]: out.append(a[i-1]); i -= 1; j -= 1
        elif L[i-1][j] >= L[i][j-1]: i -= 1
        else: j -= 1
    return ''.join(out[::-1])

def is_subseq(s, t):
    it = iter(t); return all(c in it for c in s)

for trial in range(300):
    a = ''.join(random.choice('ab') for _ in range(random.randrange(0, 10)))
    b = ''.join(random.choice('ab') for _ in range(random.randrange(0, 10)))
    s = lcs(a, b); assert is_subseq(s, a) and is_subseq(s, b)
    # brute force: longest subsequence of a that is a subsequence of b
    best = max((k for k in range(len(a) + 1) for c in itertools.combinations(a, k) if is_subseq(c, b)), default=0)
    assert len(s) == best
    m_ins_del = len(a) + len(b) - 2 * len(s)        # Exercise 5.1
print("LCS checked against brute force;", lcs("dynamic", "programming"))

def lis(x):
    T, idx, prev = [], [], [None] * len(x)          # T[l] = smallest last value of an increasing subsequence of length l+1
    for k, v in enumerate(x):
        l = bisect.bisect_left(T, v)
        if l == len(T): T.append(v); idx.append(k)
        else: T[l] = v; idx[l] = k
        prev[k] = idx[l - 1] if l else None
    out, k = [], idx[-1] if idx else None
    while k is not None: out.append(x[k]); k = prev[k]
    return out[::-1]

def lis_quadratic(x):
    best = [1] * len(x)
    for j in range(len(x)):
        for i in range(j):
            if x[i] < x[j]: best[j] = max(best[j], best[i] + 1)
    return max(best, default=0)

for trial in range(500):
    x = [random.randrange(20) for _ in range(random.randrange(0, 30))]
    s = lis(x)
    assert len(s) == lis_quadratic(x) and all(u < v for u, v in zip(s, s[1:])) and is_subseq(s, x)
print("O(n log n) LIS agrees with the quadratic DP")

For a uniformly random permutation of $n$ elements, the expected length of the longest increasing subsequence is asymptotic to $2\sqrt n$ (Logan and Shepp; Vershik and Kerov, 1977), approached from below, and Baik, Deift and Johansson (1999) proved that its fluctuations are of order $n^{1/6}$, following the Tracy–Widom law of random matrix theory. A dynamic programme is the microscope.

In [ ]:
ns = [100, 1000, 10000, 100000]; means = []
for n in ns:
    reps = 200 if n <= 10000 else 40
    vals = [len(lis(random.sample(range(n), n))) for _ in range(reps)]
    means.append(np.mean(vals))
    print(f"n={n:6d}: mean LIS {np.mean(vals):8.1f}, 2 sqrt n = {2*math.sqrt(n):7.1f}, ratio {np.mean(vals)/(2*math.sqrt(n)):.3f}, "
          f"(2 sqrt n - mean) / n^(1/6) = {(2*math.sqrt(n)-np.mean(vals))/n**(1/6):.2f}")

The last column changes slowly and approaches $1.77$: the deficit grows like $n^{1/6}$, as Baik, Deift and Johansson predict, and $-1.77$ is the mean of the Tracy–Widom distribution.

### P5.3 Knapsack

In [ ]:
def knapsack_weight(w, v, W):
    n = len(w); V = [[0] * (W + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for c in range(W + 1):
            V[i][c] = V[i-1][c]
            if w[i-1] <= c: V[i][c] = max(V[i][c], v[i-1] + V[i-1][c - w[i-1]])
    take, c = [], W
    for i in range(n, 0, -1):
        if V[i][c] != V[i-1][c]: take.append(i - 1); c -= w[i-1]
    return V[n][W], take[::-1]

def knapsack_value(w, v, W):
    n, tot = len(w), sum(v); INF = float('inf')
    G = [0] + [INF] * tot
    for i in range(n):
        for val in range(tot, v[i] - 1, -1):         # downwards: each object used once
            G[val] = min(G[val], w[i] + G[val - v[i]])
    return max(val for val in range(tot + 1) if G[val] <= W)

for trial in range(300):
    n = random.randrange(1, 12); w = [random.randint(1, 30) for _ in range(n)]; v = [random.randint(1, 30) for _ in range(n)]
    W = random.randint(0, sum(w))
    brute = max(sum(v[i] for i in S) for k in range(n + 1) for S in itertools.combinations(range(n), k) if sum(w[i] for i in S) <= W)
    best, take = knapsack_weight(w, v, W)
    assert best == brute == knapsack_value(w, v, W) and sum(w[i] for i in take) <= W and sum(v[i] for i in take) == best
print("both knapsack programmes agree with brute force")

### P5.4 Matrix chains

In [ ]:
def matrix_chain(p):
    n = len(p) - 1; M = [[0] * (n + 1) for _ in range(n + 1)]; S = [[0] * (n + 1) for _ in range(n + 1)]
    for d in range(1, n):
        for i in range(1, n - d + 1):
            j = i + d
            M[i][j], S[i][j] = min((M[i][k] + M[k+1][j] + p[i-1] * p[k] * p[j], k) for k in range(i, j))
    def paren(i, j): return f"A{i}" if i == j else f"({paren(i, S[i][j])}{paren(S[i][j] + 1, j)})"
    return M[1][n], paren(1, n)

def all_costs(p, i, j):
    if i == j: return [0]
    return [x + y + p[i-1] * p[k] * p[j] for k in range(i, j) for x in all_costs(p, i, k) for y in all_costs(p, k + 1, j)]

print(matrix_chain([10, 100, 5, 50]))
ratios = []
for trial in range(200):
    n = random.randrange(2, 9); p = [random.randint(1, 100) for _ in range(n + 1)]
    costs = all_costs(p, 1, n)
    assert matrix_chain(p)[0] == min(costs)
    assert len(costs) == math.comb(2 * n - 2, n - 1) // n          # Catalan number (Exercise 5.4)
    ratios.append(max(costs) / min(costs))
print(f"DP optimal on 200 random chains; worst/best order: median {np.median(ratios):.1f}, max {max(ratios):.0f}")

### P5.5 Optimal search trees
We use the frequencies of the 300 most common words of a Zipf distribution ($p_k\propto 1/k$), in alphabetical order (a random order of ranks).

In [ ]:
def obst_cubic(p):
    n = len(p); P = np.concatenate([[0], np.cumsum(p)]); c = [[0.0] * (n + 1) for _ in range(n + 1)]; work = 0
    for d in range(1, n + 1):
        for i in range(n - d + 1):
            j = i + d; best = math.inf
            for k in range(i + 1, j + 1):
                work += 1; best = min(best, c[i][k-1] + c[k][j])
            c[i][j] = P[j] - P[i] + best
    return c[0][n], work

def obst_knuth(p):
    n = len(p); P = np.concatenate([[0], np.cumsum(p)])
    c = [[0.0] * (n + 1) for _ in range(n + 1)]; K = [[0] * (n + 1) for _ in range(n + 1)]; work = 0
    for i in range(n): c[i][i+1] = p[i]; K[i][i+1] = i + 1
    for d in range(2, n + 1):
        for i in range(n - d + 1):
            j = i + d; best, arg = math.inf, None
            for k in range(K[i][j-1], K[i+1][j] + 1):
                work += 1; val = c[i][k-1] + c[k][j]
                if val <= best: best, arg = val, k          # '<=' keeps the largest minimiser
            c[i][j] = P[j] - P[i] + best; K[i][j] = arg
    for d in range(2, n + 1):
        for i in range(n - d + 1):
            assert K[i][i+d-1] <= K[i][i+d] <= K[i+1][i+d]
    return c[0][n], work

for trial in range(100):
    p = [random.randint(0, 20) for _ in range(random.randrange(1, 25))]
    assert abs(obst_cubic(p)[0] - obst_knuth(p)[0]) < 1e-9
print("Knuth-Yao agrees with the cubic recurrence on 100 random instances")
rng = np.random.default_rng(0)
for n in (50, 100, 200, 300):
    ranks = rng.permutation(n) + 1; p = list(1 / ranks)
    (c1, w1), (c2, w2) = obst_cubic(p), obst_knuth(p)
    print(f"n={n}: cost {c1/sum(p):.3f} comparisons per search; inner steps cubic {w1:8d}, Knuth-Yao {w2:6d} (ratio {w1/w2:.0f})")

### P5.6 The dishonest casino
The model of Durbin, Eddy, Krogh and Mitchison's *Biological Sequence Analysis* (1998): the casino switches from the fair die to the loaded one (which shows 6 half of the time) with probability $0.05$ per throw, and back with probability $0.1$. Everything is computed with logarithms to avoid underflow; for the forward–backward algorithm we normalise at each step instead.

In [ ]:
S = ['F', 'L']
start = np.log([0.5, 0.5])
A = np.log([[0.95, 0.05], [0.10, 0.90]])
Bm = np.log([[1/6] * 6, [0.1] * 5 + [0.5]])

def simulate(T, rng):
    x = [rng.choice(2, p=[0.5, 0.5])]
    for _ in range(T - 1): x.append(rng.choice(2, p=np.exp(A[x[-1]])))
    o = [rng.choice(6, p=np.exp(Bm[s])) for s in x]
    return np.array(x), np.array(o)

def viterbi(o):
    T = len(o); delta = start + Bm[:, o[0]]; back = np.zeros((T, 2), dtype=int)
    for t in range(1, T):
        cand = delta[:, None] + A            # cand[s, s'] = delta_{t-1}(s) + log a(s, s')
        back[t] = cand.argmax(axis=0); delta = cand.max(axis=0) + Bm[:, o[t]]
    x = [int(delta.argmax())]
    for t in range(T - 1, 0, -1): x.append(back[t][x[-1]])
    return np.array(x[::-1])

def posterior(o):
    T = len(o); a = np.exp(A); b = np.exp(Bm)
    alpha = np.zeros((T, 2)); beta = np.ones((T, 2)); scale = np.zeros(T)
    alpha[0] = 0.5 * b[:, o[0]]; scale[0] = alpha[0].sum(); alpha[0] /= scale[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ a) * b[:, o[t]]; scale[t] = alpha[t].sum(); alpha[t] /= scale[t]
    for t in range(T - 2, -1, -1):
        beta[t] = a @ (b[:, o[t+1]] * beta[t+1]) / scale[t+1]
    post = alpha * beta
    return post / post.sum(axis=1, keepdims=True), np.log(scale).sum()      # P(state | all rolls), log P(o)

rng = np.random.default_rng(1)
x, o = simulate(300, rng)
xv = viterbi(o); post, logp = posterior(o)
print(f"Viterbi recovers the die on {np.mean(xv == x):.0%} of throws; posterior decoding on {np.mean((post[:, 1] > 0.5) == x):.0%}; log P(rolls) = {logp:.1f}")

# brute-force check of Viterbi and the forward algorithm on short sequences
for trial in range(50):
    _, o2 = simulate(8, rng)
    joint = lambda xs: start[xs[0]] + Bm[xs[0], o2[0]] + sum(A[xs[t-1], xs[t]] + Bm[xs[t], o2[t]] for t in range(1, 8))
    allx = list(itertools.product([0, 1], repeat=8))
    assert np.isclose(joint(viterbi(o2)), max(joint(xs) for xs in allx))
    assert np.isclose(posterior(o2)[1], np.log(sum(np.exp(joint(xs)) for xs in allx)))
print("Viterbi and forward algorithm agree with brute force over all 256 hidden paths")

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.fill_between(range(len(x)), x, step='mid', color='tab:red', alpha=0.25, label='true: loaded die')
ax.plot(post[:, 1], color='tab:blue', lw=1, label='P(loaded | all rolls)')
ax.step(range(len(x)), xv * 0.95, where='mid', color='black', lw=1, label='Viterbi path')
ax.set_xlabel('throw'); ax.set_yticks([0, 1]); ax.legend(loc='upper right', fontsize=8); plt.tight_layout(); plt.show()

### P5.7 Held–Karp

In [ ]:
def held_karp(D):
    n = len(D); FULL = 1 << (n - 1)
    # city i in 1..n-1 corresponds to bit i-1; C[mask][j] for j in mask
    C = np.full((FULL, n), np.inf)
    for j in range(1, n): C[1 << (j - 1)][j] = D[0][j]
    for mask in range(1, FULL):
        for j in range(1, n):
            if not mask & (1 << (j - 1)) or mask == 1 << (j - 1): continue
            prev = mask ^ (1 << (j - 1))
            C[mask][j] = min(C[prev][i] + D[i][j] for i in range(1, n) if prev & (1 << (i - 1)))
    return min(C[FULL - 1][j] + D[j][0] for j in range(1, n))

def brute_tsp(D):
    n = len(D)
    return min(sum(D[a][b] for a, b in zip((0,) + p, p + (0,))) for p in itertools.permutations(range(1, n)))

for trial in range(40):
    n = random.randrange(2, 9); D = [[0 if i == j else random.randint(1, 50) for j in range(n)] for i in range(n)]
    assert held_karp(D) == brute_tsp(D)
print("Held-Karp agrees with brute force over all tours (asymmetric distances)")
for n in (10, 12, 14, 16):
    pts = np.random.default_rng(n).random((n, 2)); D = np.linalg.norm(pts[:, None] - pts[None], axis=2)
    t0 = time.perf_counter(); L = held_karp(D); t = time.perf_counter() - t0
    print(f"n={n}: tour length {L:.3f} in {t:5.2f} s;  (n-1)! = {math.factorial(n-1):.2e} tours, 2^n n^2 = {2**n*n*n:.2e}")

Each added city roughly doubles the time, as the theorem says; brute force would multiply it by $n$.

### P5.8 Trees

In [ ]:
def random_tree(n, rng):
    # random labelled tree from a random Pruefer sequence (Tome C1)
    if n == 1: return [[]]
    seq = [rng.randrange(n) for _ in range(n - 2)]; deg = [1] * n
    for x in seq: deg[x] += 1
    adj = [[] for _ in range(n)]
    for x in seq:
        leaf = min(i for i in range(n) if deg[i] == 1)
        adj[leaf].append(x); adj[x].append(leaf); deg[leaf] -= 1; deg[x] -= 1
    u, v = [i for i in range(n) if deg[i] == 1]
    adj[u].append(v); adj[v].append(u)
    return adj

def tree_dp(adj, wt):
    n = len(adj); order, par = [0], [None] * n; par[0] = -1
    for u in order:
        for v in adj[u]:
            if par[v] is None: par[v] = u; order.append(v)
    IN, OUT, VCIN, VCOUT = [0] * n, [0] * n, [0] * n, [0] * n
    for u in reversed(order):                       # children before parents
        ch = [v for v in adj[u] if par[v] == u]
        IN[u] = wt[u] + sum(OUT[c] for c in ch); OUT[u] = sum(max(IN[c], OUT[c]) for c in ch)
        VCIN[u] = 1 + sum(min(VCIN[c], VCOUT[c]) for c in ch); VCOUT[u] = sum(VCIN[c] for c in ch)
    return max(IN[0], OUT[0]), min(VCIN[0], VCOUT[0])

rng = random.Random(8)
for trial in range(300):
    n = rng.randrange(1, 13); adj = random_tree(n, rng); wt = [rng.randint(0, 10) for _ in range(n)]
    edges = [(u, v) for u in range(n) for v in adj[u] if u < v]
    assert len(edges) == n - 1
    subsets = [S for k in range(n + 1) for S in itertools.combinations(range(n), k)]
    mis = max(sum(wt[i] for i in S) for S in subsets if not any(u in S and v in S for u, v in edges))
    vc = min(len(S) for S in subsets if all(u in S or v in S for u, v in edges))
    assert tree_dp(adj, wt) == (mis, vc)
print("tree DPs agree with brute force on 300 random trees")

### P5.9 Breaking paragraphs

In [ ]:
text = ("Dynamic programming solves each subproblem once and stores the answer in a table, so that a recursion which "
        "would explore exponentially many overlapping branches becomes a simple sweep through polynomially many cells. "
        "The art lies in choosing the subproblems.").split()
Lw = 36

def greedy(words, L):
    lines, cur = [], []
    for w in words:
        if cur and len(' '.join(cur + [w])) > L: lines.append(cur); cur = []
        cur.append(w)
    return lines + [cur]

def optimal(words, L):
    n = len(words); f = [0] + [math.inf] * n; arg = [0] * (n + 1)
    for j in range(1, n + 1):
        width = -1
        for i in range(j, 0, -1):
            width += len(words[i-1]) + 1
            if width > L: break
            c = 0 if j == n else (L - width) ** 2
            if f[i-1] + c < f[j]: f[j], arg[j] = f[i-1] + c, i
    lines, j = [], n
    while j: lines.append(words[arg[j]-1:j]); j = arg[j] - 1
    return lines[::-1]

badness = lambda lines: sum((Lw - len(' '.join(l))) ** 2 for l in lines[:-1])
for name, lines in (("greedy", greedy(text, Lw)), ("dynamic programme", optimal(text, Lw))):
    print(f"{name} (cost {badness(lines)}):")
    for l in lines: print("  |" + ' '.join(l).ljust(Lw) + "|")